In [4]:
# ei tea kas see päriselt vajalik, lihtsalt labist võetud
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests", "psycopg2-binary", "kafka-python-ng"])
print("requests + psycopg2 + kafka-python-ng ready")

requests + psycopg2 + kafka-python-ng ready


In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *

spark = (SparkSession.builder 
  .appName("CDC-Bronze") 
  .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") 
  .config("spark.sql.catalog.lakehouse.type", "rest") 
  .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181") 
  .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.io.ResolvingFileIO")
  .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
  .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
  .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
  .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
  .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")
  .config("spark.sql.defaultCatalog", "lakehouse") 
  .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version}")

Spark 4.1.0


In [10]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.cdc")

DataFrame[]

In [ ]:
import psycopg2

PG_CONN = dict(host="postgres", port=5432, dbname="sourcedb", user="cdc_user", password="cdc_pass")

def pg_execute(sql, fetch=False):
    conn = psycopg2.connect(**PG_CONN)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(sql)
    result = cur.fetchall() if fetch else None
    cur.close()
    conn.close()
    return result

ver = pg_execute("SELECT version();", fetch=True)
print(f"PostgreSQL: {ver[0][0][:60]}...")

wal = pg_execute("SHOW wal_level;", fetch=True)
print(f"wal_level = {wal[0][0]}")
assert wal[0][0] == "logical", "wal_level must be 'logical' for CDC!"

In [1]:
pg_execute("SELECT * FROM customers ORDER BY id;", fetch=True)

NameError: name 'pg_execute' is not defined